In [1]:
import pandas as pd
import re

# Load the data
edges = pd.read_csv('edges.csv')

# 1. Clean Names & Emails
def clean_identity(text):
    if pd.isna(text): return "Unknown"
    # Remove the black blocks (█)
    text = text.replace('█', '').strip()
    # Extract name if format is "Name <email@address.com>"
    match = re.search(r'^(.*?)\s*<.*?>', text)
    if match:
        return match.group(1).strip()
    return text.strip()

edges['sender_clean'] = edges['sender'].apply(clean_identity)
edges['recipient_clean'] = edges['recipient'].apply(clean_identity)

# 2. Unify Jeffrey Epstein's Identites (Crucial for the "Aha" moments)
epstein_aliases = [
    'Jeffrey Epstein', 'J', 'jeevacation@gmail.com', 
    'jeeproject@yahoo.com', 'epstein@wanadoo.fr'
]
edges.replace({'sender_clean': epstein_aliases, 'recipient_clean': epstein_aliases}, 
              'Jeffrey Epstein', inplace=True)

# 3. Handle Dates
edges['datetime'] = pd.to_datetime(edges['datetime'], errors='coerce')

# 4. Remove purely redacted noise
# If both sender and recipient are empty/redacted, the edge is useless for network analysis
edges = edges[~((edges['sender_clean'] == "") & (edges['recipient_clean'] == ""))]

In [4]:
import networkx as nx

# Load the cleaned edge list
edges = pd.read_csv('network_edge_list.csv')

# Filter: Remove 'Redacted/Unknown' for a cleaner visualization (Barua's advice)
edges_filtered = edges[
    (edges['source'] != 'Redacted/Unknown') & 
    (edges['target'] != 'Redacted/Unknown')
]

# Create the graph
G = nx.from_pandas_edgelist(edges_filtered, source='source', target='target', edge_attr='weight')

# Now your partner can calculate Centrality and generate plots
print(f"Network has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")


Network has 1450 nodes and 1778 edges.


## Data Cleaning & Preprocessing Report
**Task:** Data Wrangling & Entity Resolution

### 1. Entity Resolution & De-duplication

The raw dataset contained significant "identity fragmentation" where the same individual appeared under multiple names, email addresses, or typos.

* **Jeffrey Epstein Consolidation:** Consolidated over 6 different identifiers (e.g., `jeeproject@yahoo.com`, `jeevacation@gmail.com`, `epstein@wanadoo.fr`, and `J`) into a single unique node: **"Jeffrey Epstein"**.
* **Name Normalization:** Standardized name formats from `Last, First` to `First Last` and removed extraneous characters, quotes, and OCR artifacts (redaction blocks `█`).
* **Handle Redactions:** Stripped noise from sender/recipient fields to extract underlying text. Where data was completely missing, it was labeled as `Redacted/Unknown` to prevent merging unrelated anonymous parties into one "super-node."

### 2. Deliverable: `cleaned_edges.csv` (The Network Engine)

This file represents the primary output of the cleaning phase. Unlike the raw message log, this is an **Aggregated Edge List** that focuses on relationships rather than individual emails.

* **Relationship Weighting:** Includes a `weight` column representing the total count of emails exchanged between two nodes. This allows for filtering by "strength of connection."
* **Temporal Tracking:** Includes `first_interaction` and `last_interaction` timestamps for every pair. This is crucial for the "Aha moments" regarding when certain individuals entered or exited the network.
* **Connectivity Mapping:** Directly links `source` and `target` names, making it ready for immediate use in `NetworkX`, `PyVis`, or `Gephi`.

### 3. Noise Reduction (Filtering)

To prevent a "hairball" effect in the visualization and focus on meaningful human interactions:

* **Newsletter/Bot Removal:** Identified and removed high-volume automated traffic (e.g., *Flipboard*, *Intelligence Squared*, and *How To Academy*).
* **Self-Loop Removal:** Filtered out instances where a sender emailed themselves.
* **Sparsity Control:** The data is structured so that the team can easily apply a "threshold filter" (e.g., `weight > 2`) to focus on core clusters.

### 4. Project Deliverables Produced

* **`cleaned_edges.csv`**: Human-readable summary of all interpersonal connections.
* **`network_edge_list.csv`**: Map-ready version using `source_id` and `target_id` for technical modeling.
* **`cleaned_nodes.csv`**: A master directory of unique entities, their IDs, and labels.